# Practical session 4: Modulating internal states with entity routines

In the last practical session, we saw how to run multiple behaviors in parallel on multiple agents. In this session we will see more advanced methods for combining behaviors by modulating their activations according to internal states of the agents. 

In practice, we will equip foraging agents with an internal "energy level" depending on the resources they consume and regulating their own behaviors.

As usual, let's first connect this notebook to the simulation:

In [ ]:
from vivarium.controllers import VivariumController
controller = VivariumController.start_session(scene_name="session_4")

The scene in this session contains three agents (the blue squares), and a number of objects with different sizes and colors (green, orange and red circles). Each of these entities is associated with a *subtype*. You can access the list of existing subtypes with:

In [ ]:
controller.subtypes

In the current scene, green objects have the subtype `"resource"`, orange objects the subtype `small_obtacle` and red objects the subtype `"big_obstacle"`. The three agents have the subtype `"agent"`.

To be all on the same page, let's first provide the definitions of the four canonical behaviors of the [Braitenberg vehicles](https://docs.google.com/presentation/d/1s6ibk_ACiJb9CERJ_8L_b4KFu9d04ZG_htUbb_YSYT4/edit#slide=id.g31e1b425a3_0_0) we have seen in class. In their definitions below we use the argument `sensed_entities=["agent"]` (see Selective sensing in [Session 3](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_3.ipynb)), meaning that these behaviors only react to agents in the scene.

In [ ]:
def fear(agent):
    left, right = agent.proximeters(sensed_entities=["agent"])
    left_wheel = left
    right_wheel = right
    return left_wheel, right_wheel

def aggression(agent):
    left, right = agent.proximeters(sensed_entities=["agent"])
    left_wheel = right
    right_wheel = left
    return left_wheel, right_wheel

def love(agent):
    left, right = agent.proximeters(sensed_entities=["agent"])
    left_wheel = 1 - left
    right_wheel = 1 - right   
    return left_wheel, right_wheel

def shyness(agent):
    left, right = agent.proximeters(sensed_entities=["agent"])
    left_wheel = 1 - right
    right_wheel = 1 - left   
    return left_wheel, right_wheel

We define a fifth behavior for obstacle avoidance. It is similar to the `shyness` behavior defined above, instead that it targets all obstacles, i.e. both the `"small_obstacle"` and `"big_obstacle"` subtypes:

In [ ]:
def obstacle_avoidance(agent):
    left, right = agent.proximeters(sensed_entities=["small_obstacle", "big_obstacle"])
    left_wheel = 1 - right
    right_wheel = 1 - left   
    return left_wheel, right_wheel

Let's also assign an alias variable to each agent of the simulation in order to access them individually in the next steps. As we saw in previous sessions, `controller.agents` is a list containing the agents and we can access its elements with Python indexing system (0 corresponding to the first element, 1 to the second etc). 

In [ ]:
agent_0 = controller.agents[0]
agent_1 = controller.agents[1]
agent_2 = controller.agents[2]

To differentiate the agents in the simulation map, we will also assign them different colors:

In [ ]:
agent_0.color = "blue"
agent_1.color = "cyan"
agent_2.color = "black"

## Weighting behaviors

As we have seen in the previous session and in the question above, it is possible to run several behaviors in parallel on the same agent. When doing it, the motor activation sent to each wheel corresponds to the average of the motor activation returned by each behavior (this averaging is implemented internally, you don't need to worry about it). 

It is also possible to specify the weight of each attached behavior, i.e. how much it will count in the averaging. This is done by passing an optional `weight` argument to the `attach_behavior` method. For example, if we want to run the `obstacle_avoidance` behavior with a weight of 1 and the `fear` behavior with a weight of 0.5 on `agent_2`, we write:

In [ ]:
agent_2.attach_behavior(obstacle_avoidance, weight=1)
agent_2.attach_behavior(fear, weight=0.5)

It might be hard to effectively see the weights in action for these two behaviors as they both lead the agent to avoid other entities. In order to better visualize the effect of the weights, we will define two new behaviors with opposite effects: `aggress_all` and `avoid_all`. The `aggress_all` behavior will make the agent accelerate towards all other entities, while the `avoid_all` behavior will make it move away from them. 

In [ ]:
def aggress_all(agent):
    left, right = agent.proximeters()  # This senses all entities
    left_wheel = right
    right_wheel = left  
    return left_wheel, right_wheel

def avoid_all(agent):
    left, right = agent.proximeters()  # This senses all entities
    left_wheel = 1 - right
    right_wheel = 1 - left   
    return left_wheel, right_wheel

We will add a larger weight to the `aggress_all` behavior than to the `avoid_all` behavior, and observe the agent's behavior:

In [ ]:
for agent in controller.agents: # For all agents
    
    # First detach all previously attached behaviors
    agent.detach_all_behaviors(stop_motors=True)

    # And attach the new behaviors with specified weights
    agent.attach_behavior(aggress_all, weight=1)
    agent.attach_behavior(avoid_all, weight=0.2)

By doing this, the wheel activations returned by the `agress_all` behavior will have more weight than those returned by the `avoid_all` behavior. For example, if `avoid_all` returns 0.6 for the left wheel, and `agress_all` returns 0.9, then the total activation of that wheel will be $(0.6 * 0.2 + 0.9 * 1) / (0.2 + 1) = 0.85$ (i.e. the average of both values weighted by their respective activation). Note that when the `weight` argument is not provided to the `attach_behavior` method, the corresponding behavior is set with a default weight of 1.

**Q1:** What are the effects of the weights on the agent's behavior? What happens if you swap the behavior weights in the cell above? What happens if both behavior have the same weight and why? Describe the observed behavior in a few lines:

*Your answer here*

## Weighting behaviors according to internal states

This weighting mechanism is particularly useful to activate a behavior according to some internal states of the agent, for example a simulated "energy level". In the following of this session we will consider the following scenario:

1. Agents forage for resources that spawn in the environment, consuming them on their way (as we did in [Session 3](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_3.ipynb))
2. Agents have a simulated "energy level", which decreases with time and increases whenever they consume a resource.
3. The weight of the foraging behavior depends on the agent's current energy level: The lower the energy level, the higher the weight. 

In consequence, the energy levels of the agents will have a tendency to naturally decrease. With lower energy levels, the agents will have a tendency to forage more for resources. Consuming resources will raise their energy level, leading to less foraging and so on and so forth.

In the following we will implement this scenario step by step.

### 1. Foraging for spawning resources.

Let's first define a `foraging` behavior (attraction towards resources):

In [ ]:
def foraging(agent):
    left, right = agent.proximeters(sensed_entities=["resource"])
    left_activation = right
    right_activation = left
    return left_activation, right_activation

Now attach only the foraging behavior to all agents:

In [ ]:
for agent in controller.agents:
    agent.detach_all_behaviors(stop_motors=True)
    agent.attach_behavior(foraging)

The agents are now attracted towards resources. However, the foraging behavior results in positive wheel activations only if the agent senses a resource, otherwise the wheel activations will be 0 (you can understand why by looking at the foraging behavior definition above). If all your agents are currently moving, drag and drop one of them such that it no longer have any resource in its field of view. In this case the agent should stay still, unless another agent pushes a resource next to it.

If we want the agents to always be in movement we need to add a behavior which results in positive wheel activations even when the agent doesn't sense any other entity. It is for instance the case of the `obstacle_avoidance` behavior, where the proximeter activations inhibits the wheel activations (resulting in a wheel at full speed if the proximeter it reads from is not activated). Let's attach the `obstacle_avoidance` behavior in addition to the foraging behavior:

In [ ]:
# Note that in the code below we do not detach the previously attached behavior
# because we want the obstacle avoid behavior to run in parrallel with the foraging behavior.

for agent in controller.agents:
    agent.attach_behavior(obstacle_avoidance)

Now agents are always in movement, attracted towards resources while avoiding obstacles. But they are not able to consume the resources yet. For this we also need to activate the consumption mechanism as we have seen in [Session 3](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_3.ipynb). The cell below will make agents consume resources whenever they are at a distance less than 1.

In [ ]:
# We first specify the "source subtype" of the consumption mechanism, 
# i.e. what subtype will be consuming other entities:
controller.consumption.source_subtype = "agent"

# Then we specify the "target subtype" of the consumption mechanism,
# i.e. what subtype is being consumed by the source subtype:
controller.consumption.target_subtype = "resource"

# Then we specify the distance range at which the consumption is triggered
controller.consumption.range = 1

# Finally we activate the consumption mechanism with:
controller.consumption.start = True

This way the agents will consume a resource whenever they are close to them. After some time all resources will have been consumed. To avoid their depletion we need to activate the spawning of resources as we have seen in [Session 3](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_3.ipynb). The cell below will spawn resources every 100 time steps: 

In [ ]:
# We first specify what subtype will be spawning
controller.spawn.subtype = "resource"

# Then we specify the time interval at which resources will spawn
# Higher values will make resources spawn less frequently
# (i.e. more time between each resource spawn)
controller.spawn.period = 100

# Finally we activate the spawning mechanism
controller.spawn.start = True

#### Knowing when an agent has consumed something

To implement the scenario outlined above, we need to continuously compute the energy level of an agent according to how many resources it has recently consumed. To do so, we need a way to know when an agent has consumed something. For this we can use the `has_consumed` method of the agent. Let's for example check if `agent_0` has consumed something:

In [ ]:
agent_0.has_consumed()

The number returned by this method indicates the number of entities that have been eaten by the agent since the last time we called it. When you first execute it, this corresponds to the number of entities consumed since the start of simulation. 
If you re-exexute the cell above a second time, it will indicate the number of entities consumed since the last time you executed it. Execute the cell above at different times while looking at how `agent_0` (the blue agent) consumes resources on the map to make sure you understand this mechanism. 

### 2. Using routines to modulate agent's internal states

To implement the scenario outlined at the beginning of this section, we need to implement a mechanism where consuming resources modulate an internal state of the agent. Here we will equip the agent with a simulated *energy level*, corresponding to a value between 0 and 1. Initially the energy level of each agent will be at 0.5, then it will slowly decrease with time when no resource is consumed, and increase whenever the agent consumes a resource. 

Let's first store the initial energy level as an agent internal state. Arbitrary internal states can be stored in `agent.internal`. For instance, to define the initial energy level of `agent_0` to 0.5 we can write:

In [ ]:
# Define an internal state called energy_level for agent_0 and set it to 0.5
agent_0.internal.energy_level = 0.5

Let's actually define the initial energy levels of all agents:

In [ ]:
for agent in controller.agents:
    agent.internal.energy_level = 0.5

We can check the value of this energy level, e.g. for `agent_0`:

In [ ]:
agent_0.internal.energy_level

If you execute the cell above several times, the `energy_level` attribute of the agent's internal state remains the same, because we haven't implemented anything to update it yet. As mentioned earlier, we want the energy level of the agent to decrease slowly with time when no resource is consumed, and increase whenever the agent consumes a resource. Thus, we need a mechanism that enable to continually modify the energy level of the agents at each time step of the simulation. The energy level of an agent will be modulated by:

- Decreasing it by a small amount (e.g. by 0.0001) at each time step of the simulation, regardless of whether or not the agent consumed a resource.
- Increasing it whenever the agent consume resources (e.g. increasing by 0.1 for each consumed resource). 
- Clipping its value between 0 and 1.

This can be implemented by attaching to agents what we call a **routine**. The definition of a *routine* is very similar to the definition of a *behavior*: It is defined as a Python function taking an agent as argument. The main difference is that a function defining a routine doesn't return any value (whereas a behavior always returns the left and right wheel activations). Thus, a routine corresponds to a set of instructions that are executed at each time step on an agent (similarly to a behavior), e.g. to compute some agent's internal states according to its interaction with the environment.

Let's define a routine called `energy` that modulates the energy level as specified above. As for a behavior, the function defining a routine takes an agent as an argument, representing the agent on which the routine will be attached:

In [ ]:
# Similarly to a behavior, a routine is defined as a function that takes an agent as argument
# The only difference is that a routine does not return anything 
# (whereas a behavior always returns the left and right wheel activations)

def energy(agent): 
    # When we will attach this routine to an agent, it will be called at every time step, 
    # and it will update the agent's energy level accordingly.

    # First we decrease the agent's energy level by a small amount,
    # regardless of whether or nor it has consumed a resource
    agent.internal.energy_level = agent.internal.energy_level - 0.0001

    # Then we read the number of resources the agent has consumed since the last time step
    number_of_resources_consumed = agent.has_consumed()

    # Then we increase the energy level according to the number of ressources the agents has consumed
    # Here we increase by 0.1 for each consumed resource
    agent.internal.energy_level = agent.internal.energy_level + 0.1 * number_of_resources_consumed

    # Finally we clip the energy level at a minimum value of 0
    if agent.internal.energy_level < 0.0:
        agent.internal.energy_level = 0.0
        
    # and at a maximum value of 1        
    if agent.internal.energy_level > 1.0:
        agent.internal.energy_level = 1.0


As for a behavior, we then need to attach the routine to an agent, which is done with the `attach_routine` method. Let's attach the `energy` routine to `agent_0`:

In [ ]:
agent_0.attach_routine(energy)

Let's check that the energy level of `agent_0` now changes according to time and the consumption of resources by executing several times the cell below:

In [ ]:
agent_0.internal.energy_level

**Q2:** How does the energy level change when `agent_0` consumes a resource ? What happens when it doesn't consume for a while ?

*Your answer here*

We can detach a routine currently attached to an agent in a similar way we detach behaviors, by calling the `detach_routine` method with the routine function as an argument. To detach the `energy` routine we do:

In [ ]:
agent_0.detach_routine(energy)

Or if we simply want to detach all the currently attached routines:

In [ ]:
agent_0.detach_all_routines()

Similarly to behaviors, we can print which routines are currently attached:

In [ ]:
agent_0.print_routines()

No routine should be attached since we have just detached it. This means that energy level is no longer modulated by the resource consumption, which we can check by executing the cell below several times (it should always print the same values, corresponding to the last energy level of the agent before we detached the `energy` routine):

In [ ]:
agent_0.internal.energy_level

Similarly to behaviors, we can attach a routine to all agents with:

In [ ]:
for agent in controller.agents:
    agent.attach_routine(energy)

Now all agents have their energy level modulated by the `energy` routine. But the energy level doesn't have any effect on the agents yet.

#### Mapping an agent internal state to a physical attribute

If we want to visualize the energy level of each agent directly in the simulation map, we can add another routine that modifies an existing physical attribute of the agent, for example its diameter, according to its energy level. To know the diameter of e.g. `agent_0` we can write:

In [ ]:
agent_0.diameter

The current diameter of the agent is 8. Let's say that we want the diameter of the agent to change between 4 and 8 according to its energy level. The diameter will be 4 when the energy level is at its minimal value (0) and it will be 8 when the energy level is at its maximum value (1), linearly interpolating between these min and max values. The formula for this is:
$$diameter = 4 + energy\_level * 4$$
Let's implement this in a new routine called `energy_diameter`:

In [ ]:
def energy_diameter(agent):
    # The diameter of the agent is proportional to its energy level
    agent.diameter = 4 + agent.internal.energy_level * 4

And attach this `energy_diameter` routine to all agents:

In [ ]:
for agent in controller.agents:
    agent.attach_routine(energy_diameter)

And that's it, now we can observe in the simulation map that the agent's diameters reflect their energy level. They become smaller when they don't consume resources for a while, and increase their diameter when they consume a resource.

As we have seen, routines can be used to modulate different attributes of an agent (or of any entity, as a routine can also be attached to objects). The attributes they modulate can be either:

- Existing attributes of the agent such as its diameter or its color.
- Custom novel attributes that you can define yourself, as we did for the energy level above.

If a routine operates on a custom attribute, this attribute has to be stored in `agent.internal` and initialized before the routine is attached. This is what we did earlier when we wrote `agent_0.internal.energy_level = 0.5`. It is strongly recommended to store custom attributes in `agent.internal`, instead of e.g. `agent.energy_level = 0.5`, to avoid conflicts with existing attributes that could potentially have the same name (this could lead to unexpected behavior, an error, or a crash). 

Moreover, it is mandatory to initialize a custom attribute before attaching a routine that reads it, otherwise the routine will raise an error because it will attempt to access an agent attribute that does not exist. This is why we previously defined the initial `energy_level` attribute (setting it to 0.5) before attaching the `energy` routine. 

### 3. Weighting a behavior according to an agent's internal state

Let's remind the scenario we outlined at the beginning of this section, that we divided in three main implementation steps:

1. Agents forage for resources that spawn in the environment, consuming them on their way.
2. Agents have a simulated "energy level", which decreases with time and increases whenever they consume a resource.
3. The weight of the foraging behavior depends on the agent's current energy level: The lower the energy level, the higher the weight.

We have implemented steps 1 and 2 above. We finally need to implement step 3, i.e. to weight the `foraging` behavior according to energy level of the agent. The lower the energy level, the higher `foraging` is weighted. 

We have seen earlier in this session how we can weight a behavior when attaching it. We have also seen how to map an agent's internal state to one of its physical attribute (mapping its energy level to its diameter). Below we will see how to map an agent's attribute to the weight of a behavior. 

In the following, we assume that all the previous code cell have been executed and that the corresponding behaviors and routines are still running. If it is not the case or if you are not sure, we recommend you quit Vivarium and reopen the current Session 4, then re-execute all the cells from the start of this notebook (as indicated in the Troubleshooting Instructions sent on Aula Global).

We can dynamically change the weight of a behavior while it is being executed on the agent with the `change_behavior_weight` method. For instance, if we want to change the weight of the `foraging` behavior of `agent_0` to 0.5 we can write:

In [ ]:
# Change the weight of the foraging behavior on agent_0 to 0.5:
agent_0.change_behavior_weight(foraging, new_weight=0.5)

Try different values of the `new_weight` argument above and check that the `agent_0` (the blue one) behaves as expected. In particular, if the weight is set to 0, you should observe that agent no longer actively forages for resources. Note that even with a weight at 0, `agent_0` might still behave as if it were foraging a bit, for instance if there is a resource on its way while it is avoiding an obstacle. If the agent is only sensing a resource though, it shouldn't be attracted by it if the behavior weight is 0. You can check this by using the Drag and Drop mechanism to isolate the blue agent on the map and place only a single resource in its field of view.

Since we want the `foraging` behavior weight to be continuously updated according the current energy level of an agent, we can define another routine for doing this. Let's call this new routine `foraging_weight`:

In [ ]:
def foraging_weight(agent):
    # This routine changes the weight of the foraging behavior according to the current energy level
    # The lower the energy level, the higher the weight (energy level is bounded between 0 and 1 in the energy routine)
    # E.g., if the energy is 1 (maximum value), the behavior weight will be 0 (and vice versa)
    weight  = 1 - agent.internal.energy_level
    agent.change_behavior_weight(foraging, new_weight=weight)

Let's attach the `foraging_weight` routine to `agent_0`:

In [ ]:
agent_0.attach_routine(foraging_weight)

Now the weight of the `foraging` behavior of `agent_0` (the blue agent) is modulated by its energy level, as implemented in the `foraging_weight` routine above. When the energy level of the blue agent is high it should be less attracted toward resources.

We can check the current weights of all attached behaviors with the `print_behaviors` method by setting the `full_infos` argument to `True`:

In [ ]:
agent_0.print_behaviors(full_infos=True)

You can re-execute the cell above several times to check that the weight of the `foraging` behavior of `agent_0` is modulated according to the energy level as expected.

### Wrapping it up
Alright, we have now fully implemented our scenario. Let's wrap it up by detaching all attached behaviors and routines, attaching both the `obstacle_avoidance` and `foraging` behaviors as well as both the `energy`, `energy_diameter` and `foraging_weight` routines, to all agents:

In [ ]:
for agent in controller.agents: # For all agents
    agent.detach_all_behaviors(stop_motors=True)  #Detach all previously attached behaviors
    agent.detach_all_routines()  #Detach all previously attached routines
    
    agent.attach_behavior(obstacle_avoidance)  # Attach the obstacle avoidance behavior
    agent.attach_behavior(foraging)  # Attach the foraging behavior
    
    agent.attach_routine(energy)  # Attach the energy routine
    agent.attach_routine(energy_diameter)  # Attach the energy_diameter routine
    
    agent.attach_routine(foraging_weight)  # Attach the foraging_weight routine    

Note that the order in which you attach routines and behaviors can matter. For instance, the `foraging_weight` routine has to be attached **after** the `foraging` behavior, because this routine changes the weight of this behavior. If the routine is attached before the behavior, it might raise an error because it will try to access a behavior that is not yet attached.

In the simulation map you should observe that all agents are indeed avoiding obstacles (red and orange circles), are attracted toward green resources and consume them. When they eat a resource their energy level increases, which we can visualize by looking at the changing agent diameters. When the energy level of an agent is high, it is less attracted toward resources, until its energy level decreases again.

**Q3:** In your mind, imagine these two different simulation conditions:

1. All agents always keep a constant weight of 1 for their foraging behavior.
2. All agents instead modulate the weight of their foraging behavior according to their energy level. (The lower their energy level, the higher the weight of their foraging behavior).

In both conditions, we consider that resources are regularly spawning in the environment and that agents are consuming them. 

What differences could you predict in the dynamics of the agent's energy levels when comparing conditions 1 vs.2? 

*Tip:* Think about the situation where two agents have their energy levels at the maximum and the third agent is at minimum energy, while resources are scarce in the environment. Does the third agent have more chance to consume a resource soon in condition 1 or in condition 2?

This is just a thought experiment, you can answer by just explaining your predictions in the markdown cell below in text form, no need to test this in simulation for now. But this kind of thought experiment could provide inspiration for your miniproject.

*Your answer here*

**Q4**: Using the mechanisms we have seen in this session, implement the following scenario:

- Agents are avoiding the big red obstacles as well as other agents.
- Resources are regularly spawning in the environment (as previously).
- Agents are foraging for resources, consuming them on their way (as previously).
- Agents have an energy level modulated by their resource consumption (as previously).
- The diameter of the agents depends on their energy level (as previously).
- The weight of the foraging behavior is modulated by the agent's energy level (as previously).
- Agents have an additional behavior that attract them toward small orange obstacles.
- The weight of the behavior for attraction toward orange obstacles is modulated by the energy level. The higher the energy level, the **more** they are attracted.

In consequence, agents should (roughly) forage for resources when their energy level is high, while pushing orange obstacles when their energy is low. At the end it boils down to implementing a couple of new behaviors and one new routine in addition to some of the ones we have already implemented in this session.

As a general guideline, when you want to implement a new scenario as the one above, or if you just want to start for scratch, start by detaching all routines and behaviors on all agents to make sure you start from a clean state. Also, feel free to copy-paste the relevant code cells from this section in code cells below, such that you can have all your code at the same place. If you want to start from the cleanest state as possible, you can just save this notebook, quit Vivarium, and reopen this session, it's pretty quick to do it (see Troubleshooting Instructions in Aula Global). 

In [ ]:
# your code here

Once you have completed this session you can save your notebook, quit Vivarium and deliver the notebook on Aula Global. 